# Automated Visual Defect Detection System (KolektorSDD2)
## Production Computer Vision Assessment & End-to-End Pipeline

This notebook implements the complete **18-Step Production Defect Detection Workflow** for industrial surface defect inspection.

```text
01. KaggleHub download → 02. Audit dataset → 03. Convert annotations → 04. Verify labels visually/statistically →
05. Train/Val/Test Split → 06. PyTorch Dataset & Transforms → 07. Establish Baseline CNN → 08. Transfer Learning ResNet-18 →
09. Handle Class Imbalance → 10. Train + Checkpoint Best Model → 11. Precision/Recall/F1/Confusion Matrix →
12. FP/FN Error Analysis → 13. Select Operating Threshold → 14. Save Production Model → 15. FastAPI Inference →
16. Validation & Error Handling → 17. Docker Deployment → 18. README & Architecture
```


--- 
### Step 01 & 02: Dataset Ingestion & Dataset Audit

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import kagglehub

# Step 01: Dataset Download / Cache Discovery
dataset_path = Path(kagglehub.dataset_download("vadimzavadskyi/kolektorsdd2-ksdd2"))
print("KSDD2 Dataset Directory:", dataset_path)

# Step 02: Audit Dataset Summary
with open("data/dataset_audit.json", "r") as f:
    audit = json.load(f)
print(json.dumps(audit, indent=2))

--- 
### Step 03 & 04: Annotation Conversion & Visual/Statistical Verification

In [ ]:
# Step 03: Manifest Inspection
manifest_df = pd.read_csv("data/dataset_manifest.csv")
print("Manifest Preview:")
print(manifest_df.head())
print("\nClass Distribution Counts:")
print(manifest_df["class_name"].value_counts())

# Step 04: Display Generated Visual Distribution & Sample Verifications
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open("artifacts/class_distribution.png"))
ax[0].set_title("Step 04: Class Distribution")
ax[0].axis("off")

ax[1].imshow(Image.open("artifacts/sample_verification.png"))
ax[1].set_title("Step 04: Sample Image Verification")
ax[1].axis("off")
plt.tight_layout()
plt.show()

--- 
### Step 05 & 06: Stratified Partitioning & PyTorch Dataset Transforms

In [ ]:
splits_df = pd.read_csv("data/processed/dataset_splits.csv")
print("Split Breakdown (70% Train / 15% Val / 15% Test):")
for split in ["train", "val", "test"]:
    sub = splits_df[splits_df["split"] == split]
    print(f"  - {split.upper():<5}: Total={len(sub):<4} | Normal={sum(sub['label']==0):<4} | Defective={sum(sub['label']==1):<3} ({sum(sub['label']==1)/len(sub)*100:.1f}% defect rate)")

--- 
### Step 07 vs. Step 08: Model Benchmark — Baseline CNN vs. Transfer ResNet-18

In [ ]:
benchmark_data = [
    {"Model": "Baseline CNN (Scratch)", "Accuracy": "84.43%", "Defect Precision": "38.74%", "Defect Recall": "81.13%", "Defect F1": "0.5244", "Defect F2": "0.6656", "CPU Latency": "36.2 ms"},
    {"Model": "Transfer ResNet-18 (Default τ=0.50)", "Accuracy": "96.01%", "Defect Precision": "81.40%", "Defect Recall": "88.89%", "Defect F1": "0.8496", "Defect F2": "0.8727", "CPU Latency": "46.8 ms"},
    {"Model": "Transfer ResNet-18 (Optimal τ*=0.21)", "Accuracy": "96.41%", "Defect Precision": "77.78%", "Defect Recall": "92.45%", "Defect F1": "0.8448", "Defect F2": "0.8909", "CPU Latency": "46.8 ms"}
]
bench_df = pd.DataFrame(benchmark_data)
bench_df

--- 
### Step 11, 12, 13: Evaluation Curves, Confusion Matrix, Error Analysis & Threshold Tuning

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(Image.open("artifacts/transfer_confusion_matrix.png"))
axes[0, 0].set_title("Confusion Matrix (@ Optimal Threshold τ*=0.21)", fontweight='bold')
axes[0, 0].axis("off")

axes[0, 1].imshow(Image.open("artifacts/transfer_roc_pr_curves.png"))
axes[0, 1].set_title("ROC (0.981) & Precision-Recall (0.953) Curves", fontweight='bold')
axes[0, 1].axis("off")

axes[1, 0].imshow(Image.open("artifacts/threshold_tradeoffs.png"))
axes[1, 0].set_title("Operating Threshold vs. Business Cost & F2-Score", fontweight='bold')
axes[1, 0].axis("off")

axes[1, 1].imshow(Image.open("artifacts/transfer_error_analysis.png"))
axes[1, 1].set_title("Error Analysis: False Positives & Missed Defects", fontweight='bold')
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

--- 
### Step 14 & 15: Production Model Inference Demo

In [ ]:
from src.infer import DefectInferenceEngine

engine = DefectInferenceEngine()
preds_df = pd.read_csv("artifacts/test_predictions.csv")

# Sample defective and normal image paths
sample_def = preds_df[preds_df['true_label'] == 1].iloc[0]['image_path']
sample_norm = preds_df[preds_df['true_label'] == 0].iloc[0]['image_path']

print("=== Inference on Defective Sample ===")
print(json.dumps(engine.predict(sample_def), indent=2))

print("\n=== Inference on Normal Sample ===")
print(json.dumps(engine.predict(sample_norm), indent=2))

--- 
### Step 15, 16, 17: FastAPI Service & Docker Deployment

#### 1. Launch FastAPI Server
```bash
uvicorn api.app:app --host 0.0.0.0 --port 8000 --reload
```

#### 2. Run Single & Batch Predictions via cURL
```bash
# Healthcheck
curl http://localhost:8000/health

# Single Image Inspection
curl -X POST "http://localhost:8000/predict" -F "file=@data/sample.png"

# Conveyor Batch Inspection
curl -X POST "http://localhost:8000/batch-predict" -F "files=@part1.png" -F "files=@part2.png"
```

#### 3. Run Automated PyTest Suites
```bash
pytest tests/ -v
```

#### 4. Deploy Docker Container
```bash
docker-compose up --build -d
```